# SNCP-PPO Social Navigation - Colab Notebook

End-to-end notebook for preparing and running the current **SNCP-PPO** experiment on Colab.

## Run order
1. **Setup** - clone repo, install deps, optional Drive mount
2. **Smoke test** - verify env + model + a tiny training loop
3. **V37 paired probe** - run C0/C1 fine-tunes from the locked v34 checkpoint
4. **Analyze** - apply the preregistered GO/NO-GO gate
5. **Visualize / inspect** - optional plots and generated reports
6. **Persist** - download probe artifacts before the session ends

## Current run: v37 - gated human-human intention graph paired probe

v36 is complete and negative/flat. It should **not** be used as a warm-start. The locked v37 base is
`v34-fixed-beta`, supplied as `sncp_ppo_v34.pt` at the repo root. V37 is not a full training run yet:
this notebook runs the preregistered **paired probe** first.

- **C0 control:** exact v34 Beta checkpoint continuation via `--init_checkpoint`.
- **C1 v37 core:** the same v34 checkpoint upgraded with zero-init gated HH self-attention + 1-4 step constant-velocity intent geometry via `--upgrade_checkpoint --hh_intent_graph`.
- Seeds: `40 41 42`; densities: `5 10 15 20`; artifacts: `eval_v37_probe/`.
- Full v37 training is allowed only if `scratch/_analyze_v37_probe.py` reports `GO`.

## Colab tips
- **Runtime -> Change runtime type -> A100** recommended.
- Upload or copy `sncp_ppo_v34.pt` into the repo root before Section 3.
- Mount Drive (Section 1.4) so checkpoints/logs survive a disconnect.


## 1. Setup

### 1.1 GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1.2 Clone / update repository

Re-run after any push to pull the latest code. Note: this updates the repo files on the VM, **not** an already-open notebook — reopen the notebook from GitHub to get notebook changes.

In [ ]:
import os
REPO_URL = 'https://github.com/heimdilon/sncp-ppo-crowdnav.git'
REPO_DIR = '/content/sncp-ppo-crowdnav'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull --rebase

%cd {REPO_DIR}
!git log --oneline -1

### 1.3 Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import torch
print(f'torch     {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'          device: {torch.cuda.get_device_name(0)}')

import gymnasium, ncps, numpy, matplotlib
print(f'gymnasium {gymnasium.__version__}')
print(f'ncps      {ncps.__version__}')
print(f'numpy     {numpy.__version__}')

### 1.4 (Optional) Mount Google Drive

Set `USE_DRIVE = True` to persist `checkpoints/` and `logs/` across sessions (recommended for long runs — a disconnect mid-training otherwise loses everything).

In [ ]:
USE_DRIVE = False  # set True to persist runs across Colab sessions
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sncp-ppo-crowdnav-runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_PROJECT_DIR}/logs', exist_ok=True)
    import shutil
    for sub in ('checkpoints', 'logs'):
        local = f'{REPO_DIR}/{sub}'
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            for f in os.listdir(local):
                dst = f'{DRIVE_PROJECT_DIR}/{sub}/{f}'
                if not os.path.exists(dst):
                    shutil.copy2(f'{local}/{f}', dst)
            shutil.rmtree(local)
        os.symlink(f'{DRIVE_PROJECT_DIR}/{sub}', local)
    print(f'Drive-backed dirs: {DRIVE_PROJECT_DIR}/{{checkpoints,logs}}')
else:
    print('Drive mount skipped (USE_DRIVE=False). Files are lost when the Colab session ends.')

## 2. Smoke tests

Fast sanity checks before spending GPU hours. Env + model first, then a 50-episode single-env training loop (legacy path) that exercises curriculum, holdout, value clipping, LR schedule, and the per-update diagnostics line.

In [ ]:
!python test_env.py

In [ ]:
!python test_model.py

In [ ]:
# 50-episode single-env smoke. NOT the full run (that's Section 3). Replay is 0
# here so this stays a quick baseline check of the single-env path.
import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--episodes', '50',
    '--num_humans', '5',
    '--seed', '42',
    '--eval_freq', '25',
    '--holdout_episodes', '3',
    '--holdout_scenarios', 'easy', 'hard',
    '--update_freq', '5',
    '--log_freq', '10',
    '--curriculum_replay_ratio', '0.0',
    '--save_path', 'checkpoints/sncp_ppo_smoke.pt',
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')

## 3. V37 paired probe (not full training)

This section runs the probe matrix defined in `scripts/run_v37_probes.py`:

| Arm | Meaning | Checkpoint path |
| --- | --- | --- |
| C0 | v34 continuation control | `checkpoints/sncp_ppo_v37_probe_c0_s*.pt` |
| C1 | v37 gated intention graph | `checkpoints/sncp_ppo_v37_probe_c1_s*.pt` |

The only intended C0/C1 difference is exact continuation vs the zero-gated v37 branch. Failed v36 levers
(node256/96, robot-human multi-head, count-scaling, sense-range, combined 4M recipe) are intentionally
not part of this probe.


### 3.1 Run probe matrix

Before running this cell, make sure `sncp_ppo_v34.pt` exists at the repo root. If it is in Drive, copy it here first.


In [ ]:
# v37 paired probe: C0 exact v34 continuation vs C1 zero-gated HH+CV intention graph.
# This is intentionally NOT the failed v36 combined-levers full run.
BASE_CHECKPOINT = 'sncp_ppo_v34.pt'
OUTPUT_DIR = 'eval_v37_probe'
TOTAL_STEPS = 300_000
EVAL_EPISODES = 100

import os, subprocess, sys

if not os.path.exists(BASE_CHECKPOINT):
    raise FileNotFoundError(
        f'{BASE_CHECKPOINT} not found. Upload/copy the locked v34-fixed-beta checkpoint '
        'to the repo root before running the v37 probe.'
    )

cmd = [
    sys.executable, '-u', 'scripts/run_v37_probes.py',
    '--mode', 'run',
    '--base_checkpoint', BASE_CHECKPOINT,
    '--output_dir', OUTPUT_DIR,
    '--python', sys.executable,
    '--eval_episodes', str(EVAL_EPISODES),
    '--total_steps', str(TOTAL_STEPS),
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print()
print(f'Exited with code {p.returncode}')
if p.returncode != 0:
    raise SystemExit(p.returncode)


### Resuming after a disconnect

There's no resume CLI. With `USE_DRIVE=True` the best checkpoint is safe in Drive; the simplest restart is to rerun 3.2 (optionally with a different `--seed`). The best-checkpoint logic keeps the highest-`min(success)` weights regardless of later collapse.

## 4. V37 probe analysis

This applies the preregistered decision rule to the paired episode bank in `eval_v37_probe/`.
Read `eval_v37_probe/report.md` and `eval_v37_probe/verdict.json` directly. A `NO-GO` result means no full
v37 training should be started; keep v34 as the base/champion.


In [ ]:
EVAL_OUT = 'eval_v37_probe'

import os, subprocess, sys, json
from IPython.display import Markdown, display

cmd = [sys.executable, 'scratch/_analyze_v37_probe.py', '--input_dir', EVAL_OUT]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd)
if result.returncode != 0:
    raise SystemExit(result.returncode)

report = os.path.join(EVAL_OUT, 'report.md')
verdict = os.path.join(EVAL_OUT, 'verdict.json')
if os.path.exists(report):
    with open(report, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
if os.path.exists(verdict):
    with open(verdict, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print('Verdict:', data.get('verdict'))


## 5. Visualize trajectories

Visualizers run in the v27 paper scenario (`paper_challenging`, robot 1.0 m/s) so
the plots reflect what the checkpoint actually learned. `CHECKPOINT` is set in
Section 4.


In [ ]:
# Single trajectory plot — first successful episode out of 20 tries.
!python scripts/visualize_trajectory.py \
    --checkpoint {CHECKPOINT} \
    --output trajectory_plot.png \
    --num_humans 10 \
    --scenario paper_challenging \
    --robot_vpref 1.0 --human_vpref_override 1.0 --human_goal_noise 0.0 --max_time 50 \
    --seed 42

from IPython.display import Image, display
display(Image('trajectory_plot.png'))

In [ ]:
# Animated GIF for a single scenario.
!python scripts/visualize_trajectory_gif.py --checkpoint {CHECKPOINT} --num_humans 10 --scenario paper_challenging

from IPython.display import Image, display
import glob
gifs = sorted(glob.glob('*.gif'))
if gifs:
    print(f'Generated: {gifs}')
    display(Image(gifs[-1]))

## 6. Training curves

Plots the newest training CSV and shows the v22 diagnostics + artifact-verification reports.

In [ ]:
import glob, os
from IPython.display import Markdown, display

if os.path.exists('eval_v37_probe/report.md'):
    with open('eval_v37_probe/report.md', 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print('eval_v37_probe/report.md not found. Run Section 4 first.')

csv_files = sorted(glob.glob('eval_v37_probe/*_training.csv'))
if csv_files:
    print('Probe training CSVs:')
    for path in csv_files:
        print(' -', path)
else:
    print('No probe training CSVs found yet.')


### Inspect CSV in pandas (optional)

In [ ]:
import pandas as pd, glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f'Rows: {len(df)}')
    print(f'Columns: {list(df.columns)}')
    print('\nPhase distribution:')
    print(df['scenario'].value_counts().sort_index())
    print('\nHoldout success per eval (event points):')
    holdout_cols = [c for c in df.columns if c.startswith('holdout_') and c.endswith('_success')]
    if holdout_cols:
        hdf = df[holdout_cols].drop_duplicates()
        hdf.index = df.loc[hdf.index, 'episode']
        print(hdf.tail(10))

## 7. Persist results

With `USE_DRIVE=True`, checkpoints/logs may already be in Drive. Otherwise set `DOWNLOAD = True` to grab the
full `eval_v37_probe` evidence bundle before the session ends.


In [ ]:
from google.colab import files
import os, shutil

DOWNLOAD = False  # set True to trigger browser download dialogs
if DOWNLOAD:
    if os.path.isdir('eval_v37_probe'):
        archive = shutil.make_archive('eval_v37_probe_artifacts', 'zip', 'eval_v37_probe')
        files.download(archive)
    else:
        print('eval_v37_probe not found. Run Sections 3 and 4 first.')


## 8. Notes & roadmap (current: v37 paired probe)

`AGENTS.md` + the v37 plan are the source of truth.

### Story so far
- **v34-fixed-beta** is the locked base: corrected Beta action distribution, best high-N result so far.
- **v36** combined all failed levers, completed the full 4M run, and was negative/flat; do not warm-start from v36.
- **v37** tests one new mechanism only: zero-init gated human-human self-attention + constant-velocity intent geometry.

### Decision rule
- Run C0/C1 paired probes for seeds 40/41/42.
- Analyze `eval_v37_probe/*_episodes.json` with `scratch/_analyze_v37_probe.py`.
- Start a full v37 run only if the report says `GO`; otherwise keep v34 and record v37 as a negative probe.
